# Avant le TP4 : observer la réplication et piloter les nœuds depuis le notebook

Le TP4 arrête un nœud du cluster pendant qu'un producteur écrit en continu. Cette démonstration montre les outils qui servent à cela, sur un topic jetable `demo-tp4`, sans arrêter de nœud :

1. lire leader, répliques et ISR de chaque partition ;
2. savoir d'où vient la valeur d'un paramètre (topic, cluster, défaut) ;
3. faire tourner un producteur en arrière-plan et suivre ses compteurs ;
4. agir sur les conteneurs du lab avec `docker compose`.

Chaque étape montre la version **terminal**, puis la version **Python**. Une étape qui ne fait que lire exécute les deux. Une étape qui modifie le cluster n'en exécute qu'une : l'autre est affichée à côté, **à ne pas lancer en plus**.

Sorties de référence : exécution du 12/11/2026 22:33 UTC sur le lab.

Les noms et les valeurs sont définis une seule fois, ici. Dans un terminal de JupyterLab, définissez d'abord : `export TOPIC=demo-tp4`.

Création du topic, équivalent en terminal (**ne pas lancer en plus**) :

```bash
docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --create --topic $TOPIC \
  --partitions 3 --replication-factor 3 --config min.insync.replicas=2
```

**En Python**

In [1]:
import os
import threading
import time

from confluent_kafka import KafkaError, KafkaException, Producer
from confluent_kafka.admin import AdminClient, ConfigResource, ConfigSource, NewTopic

BS = os.environ["BS"]
CONF = {"bootstrap.servers": BS}

TOPIC = "demo-tp4"
PARTITIONS = 3
RF = 3
MIN_ISR = 2
NUM_MESSAGES = 300
RATE = 100  # messages par seconde
SETTINGS = ("min.insync.replicas", "retention.ms", "unclean.leader.election.enable")

admin = AdminClient(CONF)
try:
    admin.create_topics([NewTopic(TOPIC, num_partitions=PARTITIONS, replication_factor=RF,
                                  config={"min.insync.replicas": str(MIN_ISR)})])[TOPIC].result()
    print(f"{TOPIC} créé")
except KafkaException as e:
    if e.args[0].code() != KafkaError.TOPIC_ALREADY_EXISTS:
        raise
    print(f"{TOPIC} existe déjà")

demo-tp4 créé


## 1. Leader, répliques et ISR

**En terminal** : une ligne par partition. L'outil Java ajoute les colonnes `Elr` et `LastKnownElr` (Eligible Leader Replicas), que le client Python ne montre pas.

In [2]:
!docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --describe --topic $TOPIC

Topic: demo-tp4	TopicId: ULRyljEYTwWh7ImXc2P8vQ	PartitionCount: 3	ReplicationFactor: 3

	Configs: min.insync.replicas=2,segment.bytes=1073741824,unclean.leader.election.enable=false
	Topic: demo-tp4	Partition: 0	Leader: 3	Replicas: 3,1,2	Isr: 3,1,2	Elr: 	LastKnownElr: 
	Topic: demo-tp4	Partition: 1	Leader: 1	Replicas: 1,2,3	Isr: 1,2,3	Elr: 	LastKnownElr: 
	Topic: demo-tp4	Partition: 2	Leader: 2	Replicas: 2,3,1	Isr: 2,3,1	Elr: 	LastKnownElr: 


**En Python** : `list_topics()` renvoie les métadonnées du cluster. Pour chaque partition : `leader`, `replicas` (le placement prévu, le premier est le leader préféré) et `isrs` (les répliques synchronisées). Une partition est **sous-répliquée** quand son ISR est plus courte que sa liste de répliques.

In [3]:
admin = AdminClient(CONF)
topic = admin.list_topics(TOPIC, timeout=10).topics[TOPIC]
if topic.error is not None:
    raise KafkaException(topic.error)

print("partition leader répliques isr        sous-répliquée")
for p in sorted(topic.partitions.values(), key=lambda p: p.id):
    print("%-9d %-6d %-9s %-12s %s" % (p.id, p.leader, p.replicas, p.isrs, len(p.isrs) < len(p.replicas)))

partition leader répliques isr        sous-répliquée
0         3      [3, 1, 2] [3, 1, 2]    False
1         1      [1, 2, 3] [1, 2, 3]    False
2         2      [2, 3, 1] [2, 3, 1]    False


## 2. D'où vient une valeur ?

**En terminal** : avec `--all`, chaque paramètre est suivi de son origine entre accolades : `DYNAMIC_TOPIC_CONFIG` (posé sur le topic), `DYNAMIC_DEFAULT_BROKER_CONFIG` (posé sur le cluster), `DEFAULT_CONFIG` (valeur par défaut).

In [4]:
!docker exec kafka-tools kafka-configs.sh --bootstrap-server $BS --describe --entity-type topics --entity-name $TOPIC --all | grep -E "^ *(min.insync.replicas|retention.ms|unclean.leader.election.enable)="

  min.insync.replicas=2 sensitive=false synonyms={DYNAMIC_TOPIC_CONFIG:min.insync.replicas=2, DYNAMIC_DEFAULT_BROKER_CONFIG:min.insync.replicas=1, DEFAULT_CONFIG:min.insync.replicas=1}
  retention.ms=604800000 sensitive=false synonyms={}
  unclean.leader.election.enable=false sensitive=false synonyms={STATIC_BROKER_CONFIG:unclean.leader.election.enable=false, DEFAULT_CONFIG:unclean.leader.election.enable=false}


**En Python** : `describe_configs()` donne, pour chaque paramètre, sa valeur et son origine (`source`).

In [5]:
admin = AdminClient(CONF)
resource = ConfigResource("topic", TOPIC)
config = admin.describe_configs([resource])[resource].result()
for name in SETTINGS:
    print("%-32s %-10s %s" % (name, config[name].value, ConfigSource(config[name].source).name))

min.insync.replicas              2          DYNAMIC_TOPIC_CONFIG
retention.ms                     604800000  DEFAULT_CONFIG
unclean.leader.election.enable   false      STATIC_BROKER_CONFIG


## 3. Un producteur en arrière-plan

Le TP4 doit garder un producteur actif pendant qu'on arrête un nœud.

En terminal, l'outil de test de charge joue ce rôle, dans un terminal à part (**ne pas lancer en plus**) :

```bash
docker exec kafka-tools kafka-producer-perf-test.sh --topic $TOPIC --num-records 300 --throughput 100 \
  --record-size 100 --producer-props bootstrap.servers=$BS acks=all
```

**En Python** : le producteur tourne dans un fil d'exécution ; la cellule rend la main et on suit ses compteurs. Le TP fournit une classe `Load` construite ainsi.

In [6]:
counters = {"confirmes": 0, "erreurs": 0}


def report(err, msg):
    counters["confirmes" if err is None else "erreurs"] += 1


def send(count, rate):
    producer = Producer({**CONF, "acks": "all", "log_level": 4})
    try:
        for i in range(count):
            producer.produce(TOPIC, value="M-%d" % i, on_delivery=report)
            producer.poll(0)
            time.sleep(1 / rate)
    finally:
        remaining = producer.flush(30)
        if remaining:
            print(remaining, "message(s) non confirmé(s)")


thread = threading.Thread(target=send, args=(NUM_MESSAGES, RATE), daemon=True)
thread.start()
for _ in range(4):
    print(counters, "en cours" if thread.is_alive() else "termine")
    time.sleep(1)

{'confirmes': 0, 'erreurs': 0} en cours


{'confirmes': 95, 'erreurs': 0} en cours


{'confirmes': 191, 'erreurs': 0} en cours


{'confirmes': 285, 'erreurs': 0} en cours


## 4. Agir sur les conteneurs avec `docker compose`

Une cellule `%%bash` lance des commandes dans le dossier `lab/` : c'est ainsi que le TP arrête puis redémarre un nœud (`docker compose stop`, `docker compose start`). Le client Python n'a pas d'équivalent : les conteneurs se pilotent en terminal. Ici, simple lecture de l'état des nœuds.

**En terminal**

In [7]:
%%bash
cd $FORMATION_DIR/lab
docker compose ps --format '{{.Name}} {{.Status}}' kafka-1 kafka-2 kafka-3

kafka-1 Up 7 minutes (healthy)
kafka-2 Up 7 minutes (healthy)
kafka-3 Up 6 minutes (healthy)


## Ménage

Équivalent en terminal (**ne pas lancer en plus**) :

```bash
docker exec kafka-tools kafka-topics.sh --bootstrap-server $BS --delete --topic $TOPIC
```

**En Python** : on attend la fin du producteur, puis on supprime le topic.

In [8]:
thread.join(30)
print(counters)

admin = AdminClient(CONF)
try:
    admin.delete_topics([TOPIC])[TOPIC].result()
    print(f"{TOPIC} supprimé")
except KafkaException as e:
    if e.args[0].code() != KafkaError.UNKNOWN_TOPIC_OR_PART:
        raise
    print(f"{TOPIC} absent")

{'confirmes': 300, 'erreurs': 0}
demo-tp4 supprimé


## La méthode du TP4

Ouvrez `tps/jour1/tp4_fiabilite_panne/enonce.ipynb`.

- Remplissez les prédictions de l'étape 6.6 **avant** d'exécuter les productions : l'intérêt du TP est de confronter votre raisonnement au cluster.
- La cellule 6.4 lance la charge puis arrête `kafka-3` : exécutez-la une seule fois, d'un seul bloc.
- Pendant la panne, les commandes en ligne de commande sont plus lentes : `$BS` contient encore l'adresse du nœud arrêté.
- Ne redémarrez `kafka-3` qu'à l'étape 6.8.